# Dot-product tests

A forward modeling code can be perfectly correct and still produce a completely
wrong FWI gradient. The adjoint-state method does not merely require *an*
adjoint simulation -- it requires the adjoint operator to be the exact
**transpose** of the discrete forward operator. Not the transpose of the
continuous equations; the transpose of the code that actually runs, including its
boundary conditions, its staggered-grid indexing and its free surface.

If the two drift apart, the forward solution stays right, the gradient still
looks plausible, and the inversion quietly converges to the wrong model. The
[finite-difference check](../Inversion/ComputingGradient.ipynb) catches this, but
it is expensive and only tests the assembled gradient at one point. The
dot-product test is the cheap, sharp instrument: it checks each operator
directly, in seconds, on a tiny grid, with no wave propagation at all.

## The identity

For any linear operator $A$ with adjoint $A^{*}$, and *any* vectors
$x$ and $y$:

$$\langle A x,\; y \rangle = \langle x,\; A^{*} y \rangle .$$

So: pick random $x$ and $y$, apply the forward kernel to $x$, apply the adjoint
kernel to $y$, and form both inner products. They must agree to machine
precision. If they do not, $A^{*}$ is not the adjoint of $A$, and the gradient
built from it is wrong.

The important property is that this makes **no reference to a correct answer**.
There is no analytical solution to compare against and no tolerance to argue
about -- either the two numbers agree to rounding error or the operator is
broken.

SeisCL ships standalone numpy reimplementations of its kernels for exactly this
purpose, in `SeisCL/tests/dot_prod_surface.py` (2D) and
`dot_prod_surface_3D.py` (3D). This notebook runs them and, more importantly,
shows what a failure looks like so the passing numbers mean something.

## Loading the shipped kernels

In [1]:
import os
import sys

import numpy as np

import SeisCL

tests_dir = os.path.join(os.path.dirname(SeisCL.__file__), "tests")
sys.path.insert(0, tests_dir)

import dot_prod_surface as dp2

print("loaded", dp2.__file__)
print("FDOH =", dp2.FDOH, " nab =", dp2.nab, " Holberg coeffs =", dp2.hc)

loaded /userdata/u/gfabien/claude/SeisCL-docs/SeisCL/tests/dot_prod_surface.py
FDOH = 2  nab = 2  Holberg coeffs = [1.1382, -0.046414]


These modules are written to be run as scripts, so the material parameters
`dot_test()` reads are defined in their `__main__` block rather than at module
level. Importing therefore requires setting them ourselves -- assigning them as
module attributes is enough for the functions to pick them up.

A small grid is all that is needed: the identity is exact for any input, so
random values on a 10x10 interior (plus the `FDOH`-wide halo the stencil needs)
exercise every code path.

In [2]:
rng = np.random.default_rng(0)
n = 10

dp2.M = rng.random((n, n))
dp2.mu = rng.random((n, n))
dp2.rho = rng.random((n, n))

print("M, mu, rho set on the module:",
      dp2.M.shape, dp2.mu.shape, dp2.rho.shape)

M, mu, rho set on the module: (10, 10) (10, 10) (10, 10)


## A relative measure

`dot_test()` returns the raw difference $\langle x, A^{*}y\rangle - \langle
Ax, y\rangle$. On its own that number is not interpretable: whether $10^{-13}$
is small depends entirely on how big the inner products themselves are. What
matters is the difference *relative* to their magnitude, so let us wrap the
shipped function to report both.

`dot_test()` returns only that difference, not the two inner products it formed,
so we reimplement the handful of lines needed to get both -- reusing the
module's kernels, which is the part that matters.

In [3]:
def inner_products(forwards, adjoints, module=dp2, seed=0):
    """Return (<Ax, y>, <x, A*y>) for the given kernel chains."""
    rng = np.random.default_rng(seed)
    FDOH = module.FDOH
    shape = [2 * FDOH + n, 2 * FDOH + n]

    def rand_field():
        a = np.zeros(shape, np.float64)
        a[FDOH:-FDOH, FDOH:-FDOH] = rng.random((n, n))
        return a

    x = [rand_field() for _ in range(5)]      # vx, vz, sxx, szz, sxz
    y = [rand_field() for _ in range(5)]

    Ax = list(x)
    for fun in forwards:
        Ax = list(fun(*Ax, module.rho, module.M, module.mu))

    Ay = list(y)
    for fun in adjoints:
        Ay = list(fun(*Ay, module.rho, module.M, module.mu))

    prod_fwd = sum(np.sum(yi * axi) for yi, axi in zip(y, Ax))
    prod_adj = sum(np.sum(xi * ayi) for xi, ayi in zip(x, Ay))
    return prod_fwd, prod_adj


def dot_check(label, forwards, adjoints, module=dp2, seed=0):
    p1, p2 = inner_products(forwards, adjoints, module, seed)
    scale = max(abs(p1), abs(p2))
    rel = abs(p2 - p1) / scale if scale else abs(p2 - p1)
    status = "PASS" if rel < 1e-10 else "FAIL"
    print("%-42s <Ax,y> = %12.6e   <x,A*y> = %12.6e   rel = %8.2e   %s"
          % (label, p1, p2, rel, status))
    return rel

## The 2D tests

Each line below tests one operator or one composition of operators. `surface` is
the free-surface kernel, `cv`/`cs` the Cerjan absorbing taper applied to
velocities and stresses, `update_v`/`update_s` the velocity and stress FD
updates, and `apply_L`/`apply_T` the symmetrising transforms that make the
propagator self-adjoint (the derivation is in the module's docstring).

In [4]:
print("--- individual kernels ---")
dot_check("free surface", [dp2.surface], [dp2.surface_adj])
dot_check("Cerjan taper", [dp2.cv, dp2.cs], [dp2.cv, dp2.cs])

print("\n--- full propagator, symmetrised ---")
dot_check("L F T", [dp2.update_v, dp2.update_s, dp2.apply_L],
                   [dp2.update_v, dp2.update_s, dp2.apply_L])
dot_check("L U2 CV U1", [dp2.update_v, dp2.cv, dp2.update_s, dp2.apply_L],
                        [dp2.update_v, dp2.cv, dp2.update_s, dp2.apply_L])

--- individual kernels ---
free surface                               <Ax,y> = 1.439359e+02   <x,A*y> = 1.439359e+02   rel = 0.00e+00   PASS
Cerjan taper                               <Ax,y> = 1.312341e+02   <x,A*y> = 1.312341e+02   rel = 0.00e+00   PASS

--- full propagator, symmetrised ---
L F T                                      <Ax,y> = 1.138036e+03   <x,A*y> = 1.138036e+03   rel = 2.00e-16   PASS
L U2 CV U1                                 <Ax,y> = 1.145918e+03   <x,A*y> = 1.145918e+03   rel = 3.97e-16   PASS


3.968412308138577e-16

The relative differences are at the level of double-precision rounding, which is
the expected result: these operators *are* each other's adjoints.

## The transforms are exact inverses

`apply_L`/`apply_Lm` and `apply_T` are not adjoint pairs but inverse pairs, so
they are checked differently -- applying both must return the input unchanged.

In [5]:
FDOH = dp2.FDOH
rng = np.random.default_rng(1)
fields = []
for _ in range(5):
    a = np.zeros([2 * FDOH + n, 2 * FDOH + n], np.float64)
    a[FDOH:-FDOH, FDOH:-FDOH] = rng.random((n, n))
    fields.append(a)

out = dp2.apply_L(*fields, dp2.rho, dp2.M, dp2.mu)
out = dp2.apply_Lm(*out, dp2.rho, dp2.M, dp2.mu)
err_L = max(np.max(np.abs(a - b)) for a, b in zip(fields, out))

out = dp2.apply_T(*fields, dp2.rho, dp2.M, dp2.mu)
out = dp2.apply_T(*out, dp2.rho, dp2.M, dp2.mu)
err_T = max(np.max(np.abs(a - b)) for a, b in zip(fields, out))

print("max |x - L^-1 L x| = %.3e   (L^-1 inverts L)" % err_L)
print("max |x - T T x|    = %.3e   (T is an involution)" % err_T)

max |x - L^-1 L x| = 8.149e-14   (L^-1 inverts L)
max |x - T T x|    = 0.000e+00   (T is an involution)


## What a failure looks like

A passing test is only meaningful if you know the test can fail. The free-surface
kernel is not self-adjoint, so using `surface` as its own adjoint -- an easy
mistake, and exactly the kind of thing that silently breaks a gradient -- should
be caught immediately.

In [6]:
print("correct adjoint:")
rel_ok = dot_check("surface / surface_adj", [dp2.surface], [dp2.surface_adj])

print("\ndeliberately wrong adjoint:")
rel_bad = dot_check("surface / surface  (WRONG)", [dp2.surface], [dp2.surface])

eps = np.finfo(np.float64).eps
print("\ncorrect pairing : rel = %.2e  (%.1f x machine epsilon)"
      % (rel_ok, rel_ok / eps))
print("wrong pairing   : rel = %.2e  (%.3g x machine epsilon)"
      % (rel_bad, rel_bad / eps))

correct adjoint:
surface / surface_adj                      <Ax,y> = 1.439359e+02   <x,A*y> = 1.439359e+02   rel = 0.00e+00   PASS

deliberately wrong adjoint:
surface / surface  (WRONG)                 <Ax,y> = 1.439359e+02   <x,A*y> = 1.548577e+02   rel = 7.05e-02   FAIL

correct pairing : rel = 0.00e+00  (0.0 x machine epsilon)
wrong pairing   : rel = 7.05e-02  (3.18e+14 x machine epsilon)


The wrong pairing fails by a wide margin. This is what makes the test valuable:
the failure is not marginal, so there is no tolerance to tune and no judgement
call to make. Either the numbers match to rounding error or they do not.

Note also what the forward simulation would have done in that situation --
nothing. `surface` is unchanged, so the modeled seismograms would be identical
and every accuracy check against an analytical solution would still pass. Only
the gradient would be wrong.

## The 3D kernels

`dot_prod_surface_3D.py` does the same for the 3D operators. It is a separate
module with its own field set, so it is run through its own `__main__` block.

In [7]:
import subprocess

out = subprocess.run([sys.executable, "dot_prod_surface_3D.py"],
                     cwd=tests_dir, capture_output=True, text=True)
print(out.stdout.strip() or out.stderr.strip()[-2000:])

Dot product for the surface kernel
-4.547473508864641e-13
Testing L is inverse of L^-1
8.115730310009894e-14
Testing T is inverse of T
0.0
Dot product for LFT
-4.547473508864641e-13
Dot product for F_s' = LSFT and F_s'^* = L F T L^-1 S^* L
-4.747562343254685e-10


## Where this fits

Three checks cover three genuinely different failure modes, and passing one says
little about the others:

| check | what it validates | cost |
|---|---|---|
| **dot-product tests** (this notebook) | the adjoint kernels are the true transpose of the forward kernels | seconds, no GPU |
| [analytical solutions](AnalyticalSolutions.ipynb) | the forward solver approximates the physics | minutes |
| [finite-difference gradient check](../Inversion/ComputingGradient.ipynb) | the assembled gradient is the derivative of the misfit | many forward runs |

The dot-product test is the one to run first when a gradient looks wrong, and the
one to re-run whenever a kernel is touched. It is also the only one of the three
that isolates *which* operator is at fault, since each is tested on its own.

If you add a new kernel with a gradient path -- a new wave equation variant, a
new boundary condition -- adding it here is what makes the resulting gradient
trustworthy.